## paper 2 (draft 1)

In [ ]:
import numpy as np
import random
from itertools import combinations

# -----------------------------
# USER INPUT
# -----------------------------
n = int(input("Enter number of users (n): "))
t = int(input("Enter threshold (t): "))
q = int(input("Enter prime modulus (q): "))
secret = int(input("Enter secret value (z): "))

# -----------------------------
# STEP 1 — SECRET
# -----------------------------
point = np.array([random.randint(1, 5) for _ in range(t-1)] + [secret])
print("\n================ SECRET =================")
print("S =", point)

# -----------------------------
# STEP 2 — SHARES
# -----------------------------
shares = []
print("\n============= BLAKLEY SHARES =============")

for i in range(n):
    coeffs = np.array([random.randint(1, 5) for _ in range(t)])
    d = int(np.dot(coeffs, point))
    shares.append((coeffs, d))

    calc = " + ".join([f"{coeffs[j]}×{point[j]}" for j in range(t)])
    print(f"\nUser {i+1}: {coeffs} · {point}")
    print(f"= {calc} = {d}")

# -----------------------------
# STEP 3 — DILITHIUM KEYS
# -----------------------------
sk_D = 2
k_const = 2
pk_D = (k_const * sk_D) % q

print("\n============= DILITHIUM KEYS =============")
print(f"sk_D = {sk_D}")
print(f"pk_D = {k_const} × {sk_D} mod {q} = {pk_D}")

# -----------------------------
# STEP 4 — MESSAGE + SIGN
# -----------------------------
coeffs, d = shares[0]
m = np.append(coeffs, d % q)[:4] % q

print("\n============= MESSAGE =============")
print("m =", m)

H = int(np.sum(m) % q)
print(f"\nH(m) = ({'+'.join(map(str,m))}) mod {q} = {H}")

sigma = (H * sk_D) % q
print(f"\nσ = {H} × {sk_D} = {H*sk_D} mod {q} = {sigma}")

# -----------------------------
# STEP 5 — KYBER SETUP
# -----------------------------
A = np.array([[2, 3], [1, 4]])
s = np.array([random.randint(1, 3), random.randint(1, 3)])
e = np.array([random.randint(0, 1), random.randint(0, 1)])

t_vec = (A @ s + e) % q

print("\n============= KYBER SETUP =============")
print("A =\n", A)
print("s =", s)
print("e =", e)

print("\nCompute t = A·s + e")
print(f"{A[0,0]}×{s[0]} + {A[0,1]}×{s[1]} + {e[0]} = {t_vec[0]}")
print(f"{A[1,0]}×{s[0]} + {A[1,1]}×{s[1]} + {e[1]} = {t_vec[1]}")

# -----------------------------
# STEP 6 — ENCRYPTION
# -----------------------------
r = np.array([1, 1])
e1 = np.array([1, 0])
e2 = 1

print("\n============= ENCRYPTION =============")

# u
u = (A.T @ r + e1) % q
print("\nu = A^T·r + e1")
print(f"= ({A[0,0]}×1 + {A[1,0]}×1 + {e1[0]}, {A[0,1]}×1 + {A[1,1]}×1 + {e1[1]})")
print("u =", u)

# k
k_raw = t_vec[0]*r[0] + t_vec[1]*r[1] + e2
k = k_raw % q
print("\nk = t^T·r + e2")
print(f"= {t_vec[0]} + {t_vec[1]} + {e2} = {k_raw} mod {q} = {k}")

# v
v = (m + k) % q
print("\nv = m + k")
for i in range(len(m)):
    print(f"{m[i]} + {k} = {v[i]}")

print("v =", v)

# -----------------------------
# STEP 7 — DECRYPTION
# -----------------------------
print("\n============= DECRYPTION =============")

k_prime_raw = s[0]*u[0] + s[1]*u[1]
k_prime = k_prime_raw % q

print(f"k' = {s[0]}×{u[0]} + {s[1]}×{u[1]} = {k_prime_raw} mod {q} = {k_prime}")

m_prime = (v - k_prime) % q

print("\nm' = v - k'")
for i in range(len(v)):
    print(f"{v[i]} - {k_prime} mod {q} = {m_prime[i]}")

print("m' =", m_prime)

# -----------------------------
# STEP 8 — NO NOISE CORRECTION (FOR DILITHIUM)
# -----------------------------
print("\n============= DIRECT VERIFICATION (NO NOISE HANDLING) =============")

m_recovered = m_prime + 1  # use directly

print("Using decrypted message m' =", m_recovered)

# -----------------------------
# STEP 9 — VERIFY SIGNATURE
# -----------------------------
H_verify = int(np.sum(m_recovered) % q)

print(f"\nH(m') = ({'+'.join(map(str,m_recovered))}) mod {q} = {H_verify}")

left = (H_verify * pk_D) % q
right = (2 * sigma) % q

print(f"\nLHS = {H_verify} × {pk_D} mod {q} = {left}")
print(f"RHS = {k_const} × {sigma} mod {q} = {right}")

if left == right:
    print("\n✔ Signature VALID")
else:
    print("\n✘ Signature INVALID (tampered or incorrect decryption)")

# -----------------------------
# STEP 10 — RECONSTRUCTION
# -----------------------------
def reconstruct(shares, t):
    for combo in combinations(shares, t):
        A_rec = np.array([c for c, _ in combo], dtype=float)
        b_rec = np.array([d for _, d in combo], dtype=float)

        if abs(np.linalg.det(A_rec)) > 1e-6:
            return np.linalg.solve(A_rec, b_rec)

    return None

solution = reconstruct(shares, t)

print("\n============= RECONSTRUCTION =============")

if solution is not None:
    print("Solution =", solution)
    print("Recovered secret =", solution[-1])
else:
    print("No valid set found")

: 

## GRAPH(Noise Level,Accuracy of Recovery,Noise Impact on Recovery Accuracy)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time
import random

# -----------------------------
# SIMULATION PARAMETERS
# -----------------------------
q = 17
share_sizes = [2, 3, 4, 5, 6, 7, 8]

enc_times = []
dec_times = []
sign_times = []
verify_times = []

# -----------------------------
# SIMULATE PERFORMANCE
# -----------------------------
for n in share_sizes:
    m = np.random.randint(1, 10, size=4)

    # ---- Encryption ----
    start = time.time()
    for _ in range(1000):
        k = random.randint(1, 10)
        v = (m + k) % q
    enc_times.append((time.time() - start) * 1000)

    # ---- Decryption ----
    start = time.time()
    for _ in range(1000):
        k = random.randint(1, 10)
        m_prime = (v - k) % q
    dec_times.append((time.time() - start) * 1000)

    # ---- Signing ----
    start = time.time()
    for _ in range(1000):
        H = np.sum(m) % q
        sigma = (H * 2) % q
    sign_times.append((time.time() - start) * 1000)

    # ---- Verification ----
    start = time.time()
    for _ in range(1000):
        check = (H * 4) % q
    verify_times.append((time.time() - start) * 1000)

# -----------------------------
# GRAPH 1: TIME PERFORMANCE
# -----------------------------
plt.figure()
plt.plot(share_sizes, enc_times)
plt.plot(share_sizes, dec_times)
plt.plot(share_sizes, sign_times)
plt.plot(share_sizes, verify_times)

plt.xlabel("Number of Shares")
plt.ylabel("Time (ms)")
plt.title("Encryption / Decryption / Signing / Verification Time")
plt.legend(["Encryption", "Decryption", "Signing", "Verification"])
plt.grid()
plt.show()


# -----------------------------
# GRAPH 2: COMMUNICATION OVERHEAD (BAR GRAPH)
# -----------------------------
import numpy as np
import matplotlib.pyplot as plt

shares = np.array(share_sizes)

# Simulated overhead units
blakley = shares * 4
kyber = shares * 6
dilithium = shares * 8

x = np.arange(len(shares))
width = 0.25

plt.figure()

plt.bar(x - width, blakley, width, label="Blakley")
plt.bar(x, kyber, width, label="Blakley + Kyber")
plt.bar(x + width, dilithium, width, label="Blakley + Kyber + Dilithium")

plt.xlabel("Number of Shares")
plt.ylabel("Communication Overhead (Units)")
plt.title("Communication Overhead Comparison")

plt.xticks(x, shares)
plt.legend()
plt.grid(axis='y')

plt.show()


# -----------------------------
# GRAPH 3: NOISE vs ACCURACY
# -----------------------------
noise_levels = np.linspace(0, 5, 10)
accuracy = []

for noise in noise_levels:
    correct = 0
    trials = 100

    for _ in range(trials):
        m = np.array([2, 1, 3, 5])
        noisy = m + np.random.randint(0, int(noise)+1, size=4)

        # median rule
        median = np.sort(noisy)[len(noisy)//2]
        count = np.sum(noisy >= median)

        if count >= len(noisy)/2:
            corrected = noisy - 1
        else:
            corrected = noisy

        if np.all(corrected == m):
            correct += 1

    accuracy.append(correct / trials)

plt.figure()
plt.plot(noise_levels, accuracy)

plt.xlabel("Noise Level")
plt.ylabel("Accuracy of Recovery")
plt.title("Noise Impact on Recovery Accuracy")
plt.grid()
plt.show()

## Blakley Secret Sharing (3 Planes Intersection)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

# -----------------------------
# SECRET POINT
# -----------------------------
x0, y0, z0 = 2, 3, 5  # secret point

# -----------------------------
# DEFINE PLANES
# -----------------------------
# Plane 1: x + y + z = 10
# Plane 2: 2x + y + 3z = 22
# Plane 3: x + 2y + 3z = 23

x = np.linspace(0, 5, 10)
y = np.linspace(0, 5, 10)
X, Y = np.meshgrid(x, y)

Z1 = 10 - X - Y
Z2 = (22 - 2*X - Y) / 3
Z3 = (23 - X - 2*Y) / 3

# -----------------------------
# PLOT
# -----------------------------
fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')

# Plot planes
ax.plot_surface(X, Y, Z1, alpha=0.5)
ax.plot_surface(X, Y, Z2, alpha=0.5)
ax.plot_surface(X, Y, Z3, alpha=0.5)

# Plot intersection point
ax.scatter(x0, y0, z0, s=100)
ax.text(x0, y0, z0, " Secret (2,3,5)")

# Labels
ax.set_xlabel("X")
ax.set_ylabel("Y")
ax.set_zlabel("Z")

ax.set_title("Blakley Secret Sharing (3 Planes Intersection)")

plt.show()